In [1]:
#Import libraries
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt 
sb.set() 
import re

In [2]:
#train set(training ml models)
train_data = pd.read_csv('train.csv')
train_data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
#test set (to make predictions)
test_data = pd.read_csv("test.csv")
test_data.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [4]:
#test_labels set (evaluating performance)
testlabels_data = pd.read_csv("test_labels.csv")
testlabels_data.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1


In [5]:
# train data stats
train_data.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
# test data stats
test_data.describe()

,id,comment_text
count,153164,153164
unique,153164,153164
top,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
freq,1,1


In [7]:
# train dataset info
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [8]:
# test dataset info
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  object
 1   comment_text  153164 non-null  object
dtypes: object(2)
memory usage: 2.3+ MB


In [9]:
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (159571, 8)
Test shape: (153164, 2)


In [10]:
print("\nTrain columns:")
print(train_data.columns)

print("\nTest columns:")
print(test_data.columns)


Train columns:
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

Test columns:
Index(['id', 'comment_text'], dtype='object')


# DATA PREPROCESSING/CLEANING 
### Only train_data
- Text Cleaning✔️
- Filling missing values✔️
- TF-IDF with removal of stopwards ✔️ 
- Stemming(impacts TF-IDF performance)❌

In [11]:
#create a text cleaning function to clean raw text

def clean_text(text):
    text = str(text)
    text = text.lower()

    # remove html tags
    text = re.sub(r'<.*?>', ' ', text)

    # expand contractions
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "can not ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r"\'scuse", " excuse ", text)

    # remove number-containing tokens
    text = re.sub(r'\b\w*\d+\w*\b', ' ', text)

    # keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [12]:
#apply text cleaning on train data
train_data["clean_comment"] = train_data["comment_text"].apply(clean_text)

In [13]:
#cleaned textual train data
train_data[["comment_text", "clean_comment"]].head(10)

,comment_text,clean_comment
0,Explanation\nWhy the edits made under my usern...,explanation why the edits made under my userna...
1,D'aww! He matches this background colour I'm s...,d aww he matches this background colour i am s...
2,"Hey man, I'm really not trying to edit war. It...",hey man i am really not trying to edit war it ...
3,"""\nMore\nI can't make any real suggestions on ...",more i can not make any real suggestions on im...
4,"You, sir, are my hero. Any chance you remember...",you sir are my hero any chance you remember wh...
5,"""\n\nCongratulations from me as well, use the ...",congratulations from me as well use the tools ...
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,cocksucker before you piss around on my work
7,Your vandalism to the Matt Shirvington article...,your vandalism to the matt shirvington article...
8,Sorry if the word 'nonsense' was offensive to ...,sorry if the word nonsense was offensive to yo...
9,alignment on this subject and which are contra...,alignment on this subject and which are contra...


In [14]:
#remove null values/missing comments in case of any
train_data = train_data.dropna(subset=["clean_comment"])
train_data = train_data[train_data["clean_comment"].str.strip() != '']

In [15]:
#Final check for any null/missing values
print("Empty comments after cleaning:",
      (train_data["clean_comment"] == "").sum())

Empty comments after cleaning: 0


In [16]:
#Vocabulary check of text data
sample_words = " ".join(train_data["clean_comment"].head(1000)).split()
print(sample_words[:50])

#saving cleaned train textual data in new csv file
train_data.to_csv("train_cleaned.csv", index=False)

['explanation', 'why', 'the', 'edits', 'made', 'under', 'my', 'username', 'hardcore', 'metallica', 'fan', 'were', 'reverted', 'they', 'were', 'not', 'vandalisms', 'just', 'closure', 'on', 'some', 'gas', 'after', 'i', 'voted', 'at', 'new', 'york', 'dolls', 'fac', 'and', 'please', 'do', 'not', 'remove', 'the', 'template', 'from', 'the', 'talk', 'page', 'since', 'i', 'am', 'retired', 'now', 'd', 'aww', 'he', 'matches']


In [17]:
#Separation of text and labels to perfrom tf-idf
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

X_train = train_data["clean_comment"]   #tf-idf will be only done on comments, not on labels. 
y_train = train_data[label_cols]

In [18]:
#Performing TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
custom_stopwords = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"}

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    stop_words=list(custom_stopwords),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)

X_train_tfidf = tfidf.fit_transform(X_train) #fits the model and transforms some data

In [19]:
#viewing TF-IDF results
print("X_train_tfidf shape:", X_train_tfidf.shape)  

print("\nFirst 20 TF-IDF features:")
print(tfidf.get_feature_names_out()[:20])    # features/words/n-grams from train set (these are columns in tf-idf file)
print('\n')
feature_names = tfidf.get_feature_names_out()  # its corresponding tdf-idf numbers for the features
df_tfidf = pd.DataFrame(
    X_train_tfidf[:5].toarray(),   
    columns=feature_names)
print(df_tfidf)

X_train_tfidf shape: (159560, 30000)

First 20 TF-IDF features:
['aa' 'aap' 'aaron' 'ab' 'aba' 'abandon' 'abandoned' 'abbas' 'abbey'
 'abbreviated' 'abbreviation' 'abbreviations' 'abc' 'abc news' 'abd'
 'abdul' 'abdullah' 'abe' 'abide' 'abiding']


    aa  aap  aaron   ab  aba  abandon  abandoned  abbas  abbey  abbreviated  \
0  0.0  0.0    0.0  0.0  0.0      0.0        0.0    0.0    0.0          0.0   
1  0.0  0.0    0.0  0.0  0.0      0.0        0.0    0.0    0.0          0.0   
2  0.0  0.0    0.0  0.0  0.0      0.0        0.0    0.0    0.0          0.0   
3  0.0  0.0    0.0  0.0  0.0      0.0        0.0    0.0    0.0          0.0   
4  0.0  0.0    0.0  0.0  0.0      0.0        0.0    0.0    0.0          0.0   

   ...  zoe  zombie  zone  zones  zoo  zoom  zora   zu  zuck  zuckerberg  
0  ...  0.0     0.0   0.0    0.0  0.0   0.0   0.0  0.0   0.0         0.0  
1  ...  0.0     0.0   0.0    0.0  0.0   0.0   0.0  0.0   0.0         0.0  
2  ...  0.0     0.0   0.0    0.0  0.0   0.0   0.0  

In [20]:
#viewing features with highest tf-idf values
row = 0  # choose a document
feature_names = tfidf.get_feature_names_out()
row_values = X_train_tfidf[row].toarray().flatten()

df = pd.DataFrame({
    'feature': feature_names,
    'tfidf': row_values
})

# show only non-zero values
df = df[df['tfidf'] > 0]

# sort by importance
df = df.sort_values(by='tfidf', ascending=False)
print(df.head(10))

               feature     tfidf
7478             dolls  0.280235
15598        metallica  0.272121
28131       vandalisms  0.269250
22113  remove template  0.267487
4530           closure  0.242161
22576     reverted not  0.239823
10971         hardcore  0.236787
26046    template talk  0.227962
22513          retired  0.215697
10073              gas  0.214891


#### Variables to be used for Logistic Regression
X_train_tfidf and y_train will be used to train the Logistic Regression model.

test_data for making final predictions and test_labels is for evaluating performance